In [1]:
import torch
import torch.nn as nn

# --- Generator (U-Net) ---
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, down=True, use_dropout=False):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False, padding_mode="reflect")
            if down else
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU() if not down else nn.LeakyReLU(0.2)
        )
        self.use_dropout = use_dropout
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.conv(x)
        return self.dropout(x) if self.use_dropout else x

class Generator(nn.Module):
    def __init__(self, in_channels=3, features=64):
        super().__init__()
        # Simplified U-Net Encoder/Decoder
        self.initial_down = nn.Sequential(
            nn.Conv2d(in_channels, features, 4, 2, 1, padding_mode="reflect"),
            nn.LeakyReLU(0.2),
        )
        self.down1 = UNetBlock(features, features * 2, down=True)
        self.up1 = UNetBlock(features * 2, features, down=False)
        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(features * 1, in_channels, 4, 2, 1),
            nn.Tanh(), # Pixels normalized between -1 and 1
        )

    def forward(self, x):
        d1 = self.initial_down(x)
        d2 = self.down1(d1)
        u1 = self.up1(d2)
        return self.final_up(u1)

# --- Loss Function ---
# Pix2Pix uses a combination of Adversarial Loss and L1 Loss
# Total Loss = Loss_GAN + lambda * Loss_L1